# 05 - Statistique inferentielle : le notebook

> **Formation Data Analyst - Mathematiques pour la donnee**

Ce notebook accompagne le cours [`01-statistique-inferentielle.md`](./01-statistique-inferentielle.md). On y met les mains dans le vrai jeu de donnees **`ventes_magasins.csv`** de la Ch'ti Boutique pour pratiquer :

1. **Population vs echantillon** : deviner le tout a partir d'un petit groupe.
2. **Intervalle de confiance** sur une moyenne et sur une proportion.
3. **Test d'hypothese** et **A/B test** : vrai effet ou hasard ?
4. **Interpreter correctement une p-value** (le piege n1 du metier).

> Rappel des 3 mnemos du chapitre : **IC = la fourchette ou se cache la vraie valeur** ; **p-value < 5% = trop beau pour etre un hasard** ; la p-value n'est **PAS** la probabilite que H0 soit vraie.

In [ ]:
# Cellule 1 : imports + chargement du VRAI dataset
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint, proportions_ztest

# Le fichier de ventes de la Ch'ti Boutique (chemin relatif au notebook)
df = pd.read_csv("../../../99-Brief/Data-Analyst/data/ventes_magasins.csv", parse_dates=["date"])

print("Dimensions :", df.shape)
df.head()

## 1. Population vs echantillon

Ici, notre **population** = les **12 000 ventes** enregistrees (c'est TOUT ce qu'on possede). Dans la vraie vie, tu n'as souvent qu'un **echantillon** : un sondage, un mois de donnees, un panel.

Pour t'entrainer, on va faire semblant : on tire un **petit echantillon aleatoire** de 200 ventes et on regarde a quel point sa moyenne (x barre) s'approche de la vraie moyenne de la population (mu). Si on tirait un autre echantillon, on obtiendrait une moyenne un peu differente : c'est la **fluctuation d'echantillonnage**.

| | Population | Echantillon |
|---|---|---|
| Taille | N = 12 000 | n = 200 |
| Moyenne du montant | mu (inconnue en vrai) | x barre (calculee) |

In [ ]:
# Population complete : la vraie moyenne du montant (le mu qu'on cherche a deviner)
mu_population = df["montant"].mean()
print(f"Population : N = {len(df)}  |  moyenne reelle mu = {mu_population:.2f} EUR")

# On tire UN echantillon aleatoire de 200 ventes (random_state pour reproductibilite)
echantillon = df["montant"].sample(n=200, random_state=42)
x_barre = echantillon.mean()
print(f"Echantillon : n = {len(echantillon)}  |  moyenne observee x_barre = {x_barre:.2f} EUR")

print(f"\nEcart entre l'estimation et la verite : {x_barre - mu_population:+.2f} EUR")
print("-> x_barre n'est jamais EXACTEMENT mu : c'est l'erreur d'echantillonnage.")

## 2. Intervalle de confiance sur une MOYENNE

Plutot que d'annoncer une seule valeur (x barre), on annonce une **fourchette** qui a 95% de chances (au sens de la methode) de contenir la vraie moyenne mu.

Pour une moyenne, on utilise la **loi de Student (t)** car on ne connait pas l'ecart-type de la population :

$$ IC_{95\%} = \bar{x} \pm t_{0{,}975,\,n-1} \times \frac{s}{\sqrt{n}} $$

ou `s / sqrt(n)` est l'**erreur standard** (SE).

In [ ]:
# IC 95% sur la moyenne du montant, calcule a partir de notre echantillon de 200 ventes
n = len(echantillon)
s = echantillon.std(ddof=1)          # ecart-type de l'echantillon (ddof=1 !)
se = s / np.sqrt(n)                    # erreur standard

ic_bas, ic_haut = stats.t.interval(0.95, df=n - 1, loc=x_barre, scale=se)

print(f"Moyenne estimee : {x_barre:.2f} EUR")
print(f"IC 95% : [{ic_bas:.2f} ; {ic_haut:.2f}] EUR")
print(f"La vraie moyenne mu = {mu_population:.2f} est-elle dans l'intervalle ? "
      f"{'OUI' if ic_bas <= mu_population <= ic_haut else 'NON'}")
print("\nLecture metier : 'On est confiants a 95% que le montant moyen se situe dans cette fourchette.'")

## 3. Intervalle de confiance sur une PROPORTION

Le cas le plus frequent en sondage (taux de satisfaction, taux de clic...). Formule :

$$ IC_{95\%} = \hat{p} \pm 1{,}96 \times \sqrt{\frac{\hat{p}(1-\hat{p})}{n}} $$

**Question metier :** quelle proportion de nos ventes beneficient d'une **remise** ? On l'estime a partir d'un echantillon de 500 ventes.

In [ ]:
# Variable binaire : la vente a-t-elle une remise ? (True/False)
a_une_remise = (df["remise"] > 0)
p_population = a_une_remise.mean()   # la vraie proportion (on triche pour verifier)

# Echantillon de 500 ventes
ech_prop = a_une_remise.sample(n=500, random_state=1)
k = int(ech_prop.sum())              # nombre de ventes avec remise dans l'echantillon
n_prop = len(ech_prop)
p_hat = k / n_prop

# Methode 1 : a la main (comme dans le cours)
marge = 1.96 * np.sqrt(p_hat * (1 - p_hat) / n_prop)
print(f"p_hat = {p_hat:.1%}  |  marge d'erreur = +/-{marge:.1%}")
print(f"IC 95% (a la main)    : [{p_hat - marge:.1%} ; {p_hat + marge:.1%}]")

# Methode 2 : avec statsmodels (recommandee en production)
bas, haut = proportion_confint(count=k, nobs=n_prop, alpha=0.05, method="normal")
print(f"IC 95% (statsmodels)  : [{bas:.1%} ; {haut:.1%}]")
print(f"\nVraie proportion sur la population : {p_population:.1%} (bien dans l'intervalle)")

### 🎯 A toi de jouer (1/3) - IC sur une proportion

On veut estimer la part des ventes realisees **en E-commerce** (colonne `type`).
Complete le code pour tirer un echantillon de **400** ventes et calculer l'**IC 95%** de cette proportion. Compare ensuite a la vraie valeur sur la population.

In [ ]:
# 🎯 A TOI DE JOUER (1/3)
# Objectif : IC 95% de la proportion de ventes en E-commerce

# 1) Cree la variable binaire (True si type == 'E-commerce')
est_ecommerce = ...  # TODO : (df["type"] == "E-commerce")

# 2) Tire un echantillon de 400 lignes (utilise random_state=7)
ech_ab = ...  # TODO : est_ecommerce.sample(n=400, random_state=7)

# 3) Calcule k (nombre de True) et n, puis l'IC avec proportion_confint
# k = int(ech_ab.sum())
# n_ab = len(ech_ab)
# bas, haut = proportion_confint(count=..., nobs=..., alpha=0.05, method="normal")
# print(f"IC 95% : [{bas:.1%} ; {haut:.1%}]  |  vraie valeur : {est_ecommerce.mean():.1%}")

## 4. Test d'hypothese : vrai effet ou hasard ?

On compare le **montant moyen** entre les ventes **avec remise** et **sans remise**. Deux hypotheses :

- **H0** : les deux montants moyens sont egaux (la difference observee est due au hasard).
- **H1** : les deux montants moyens sont differents.

On utilise un **test t de Student** (`scipy.stats.ttest_ind`) sur deux echantillons independants.

> Regle de decision : **p-value < 0,05** -> on rejette H0 (difference significative).

Juste apres, on enchaine sur un **A/B test** : meme logique, mais on compare cette fois un **taux** (proportion de ventes avec remise) entre **Mode** (groupe A) et **Sport** (groupe B), avec un **test z de proportions** (`statsmodels.proportions_ztest`).

In [ ]:
avec_remise = df.loc[df["remise"] > 0, "montant"]
sans_remise = df.loc[df["remise"] == 0, "montant"]

print(f"Montant moyen AVEC remise : {avec_remise.mean():.2f} EUR (n={len(avec_remise)})")
print(f"Montant moyen SANS remise : {sans_remise.mean():.2f} EUR (n={len(sans_remise)})")

# Test t de Welch (equal_var=False : on ne suppose pas les variances egales)
t_stat, p_value = stats.ttest_ind(avec_remise, sans_remise, equal_var=False)
print(f"\nStatistique t = {t_stat:.3f}")
print(f"p-value = {p_value:.6f}")

if p_value < 0.05:
    print("\n-> p < 0,05 : SIGNIFICATIF. On rejette H0 : la difference est reelle (pas du hasard).")
else:
    print("\n-> p >= 0,05 : NON significatif. On ne peut pas conclure a une difference.")

In [ ]:
mode = df[df["categorie"] == "Mode"]
sport = df[df["categorie"] == "Sport"]

# Nombre de ventes avec remise et effectif total par groupe
conversions = np.array([(mode["remise"] > 0).sum(), (sport["remise"] > 0).sum()])
effectifs   = np.array([len(mode), len(sport)])

taux = conversions / effectifs
print(f"Taux de remise  Mode (A) : {taux[0]:.1%}  |  Sport (B) : {taux[1]:.1%}")

z_stat, p_value_ab = proportions_ztest(count=conversions, nobs=effectifs)
print(f"\nStatistique z = {z_stat:.3f}")
print(f"p-value = {p_value_ab:.4f}")

if p_value_ab < 0.05:
    print("\n-> SIGNIFICATIF : les taux de remise different vraiment entre les deux categories.")
else:
    print("\n-> NON significatif : l'ecart observe est compatible avec le hasard. On ne tranche pas.")

## 5. Interpreter correctement la p-value (le piege mortel)

> ⚠️ La p-value n'est **PAS** "la probabilite que H0 soit vraie". Elle *part du principe* que H0 est vraie, puis mesure a quel point tes donnees seraient **surprenantes** dans ce monde-la.

Verifions cette intuition : si H0 est VRAIE (aucune difference), la p-value devrait etre **uniforme entre 0 et 1**. On compare deux echantillons tires de la MEME population (donc H0 est vraie par construction), 2000 fois.

In [ ]:
# H0 vraie : on tire les DEUX groupes dans la meme colonne 'montant'
p_values = []
rng = np.random.default_rng(0)
montants = df["montant"].values
for _ in range(2000):
    a = rng.choice(montants, size=150, replace=False)
    b = rng.choice(montants, size=150, replace=False)
    _, p = stats.ttest_ind(a, b, equal_var=False)
    p_values.append(p)
p_values = np.array(p_values)

plt.figure(figsize=(8, 4))
plt.hist(p_values, bins=20, color="#55A868", edgecolor="white")
plt.axvline(0.05, color="red", linestyle="--", label="seuil 0,05")
plt.title("Distribution des p-values quand H0 est VRAIE (aucune difference)")
plt.xlabel("p-value")
plt.ylabel("Frequence")
plt.legend()
plt.show()

faux_positifs = (p_values < 0.05).mean()
print(f"Part de tests 'significatifs' alors que H0 est vraie : {faux_positifs:.1%}")
print("-> C'est environ 5% : ce sont des FAUX POSITIFS. Meme sans effet, on 'trouve' du significatif 1 fois sur 20.")
print("Morale : ne teste pas 50 variables au hasard en esperant en trouver une 'significative'.")

### 🎯 A toi de jouer (2/3) - Un test d'hypothese sur une moyenne

Le montant moyen des ventes **En ligne** (`ville == "En ligne"`) est-il different de celui des ventes de **Lille** ?
Complete le test t et conclus en fonction de la p-value.

In [ ]:
# 🎯 A TOI DE JOUER (2/3)
# H0 : montant moyen 'En ligne' == montant moyen 'Lille'
# H1 : ils sont differents

en_ligne = ...  # TODO : df.loc[df["ville"] == "En ligne", "montant"]
lille = ...     # TODO : df.loc[df["ville"] == "Lille", "montant"]

# t_stat, p_value = stats.ttest_ind(en_ligne, lille, equal_var=False)
# print(f"Moyennes : En ligne = {en_ligne.mean():.2f} | Lille = {lille.mean():.2f}")
# print(f"p-value = {p_value:.4f}")
# TODO : ecris la phrase de conclusion selon que p < 0.05 ou non

### 🎯 A toi de jouer (3/3) - Redige l'interpretation metier

Reprends le resultat de l'A/B test Mode vs Sport (section 5). Dans la cellule ci-dessous (en commentaire), **corrige** l'affirmation fausse d'un collegue :

> "La p-value vaut 0,71, donc il y a 71% de chances que les deux categories aient le meme taux de remise."

Ecris la formulation **correcte** de ce que dit reellement cette p-value, puis la **decision** a prendre.

In [ ]:
# 🎯 A TOI DE JOUER (3/3)
# Corrige l'affirmation fausse ci-dessous en completant les commentaires.

# FAUX : "p = 0,71 => 71% de chances que les taux soient egaux."
#
# TODO 1 - Reformulation correcte de la p-value :
# La p-value est la probabilite d'observer un ecart au moins aussi grand que celui mesure,
# SI ... (complete : si H0 etait vraie, c-a-d si les taux etaient reellement egaux).
#
# TODO 2 - Decision :
# Comme p = 0,71 >= 0,05, on ... (complete : on ne rejette PAS H0 -> aucune preuve de difference).

print("Ecris tes deux reponses dans les commentaires ci-dessus.")

## Synthese - ce que tu as pratique

- **Population vs echantillon** : la moyenne d'un echantillon (x barre) approche la vraie moyenne (mu) sans jamais l'egaler -> **erreur d'echantillonnage** (fluctuation d'un tirage a l'autre).
- **Intervalle de confiance** : on annonce une **fourchette**, jamais un chiffre sec. Sur une **moyenne** (loi de Student, `stats.t.interval`), sur une **proportion** (`proportion_confint`). *IC = la fourchette ou se cache la vraie valeur.* 🍴
- **Test d'hypothese** : H0 ("rien") vs H1 ("quelque chose"). `ttest_ind` pour des moyennes, `proportions_ztest` pour un A/B test de taux. **p < 0,05 -> on rejette H0**.
- **p-value bien interpretee** : c'est la probabilite d'un resultat aussi extreme **SI H0 est vraie** ; ce n'est **PAS** la probabilite que H0 soit vraie. Sous H0, on obtient ~5% de faux positifs -> mefiance quand on teste beaucoup de variables.

> **Le reflexe d'or a garder :** un resultat "significatif" ne dit ni que l'effet est **gros**, ni qu'il y a **causalite**. Et une correlation ne prouve jamais une cause a effet (revois la fin du cours : variable confondante !).

**Pour aller plus loin :** refais l'A/B test avec des sous-echantillons plus petits et observe comment la p-value devient instable quand n diminue.